# 06 — Video Inference
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives

1. Run the PPE detection model on a **video file**.
2. Overlay **bounding boxes, class labels, confidence, and FPS**.
3. Apply the **SAFE / UNSAFE** compliance overlay per frame.
4. Save the annotated video to `outputs/videos/`.

---

> **How to get a test video:**  
> - Use any construction site video (MP4/AVI).
> - Or download a sample from the original dataset source (YouTube links in `README.dataset.txt`).
> - Place the video file anywhere and update `VIDEO_PATH` below.

## 1. Setup

In [ ]:
import sys, time
from pathlib import Path

import cv2

_nb_dir = Path().resolve()
PROJECT_ROOT = _nb_dir
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ppe_detection.utils import MODELS_DIR, OUTPUTS_DIR, ensure_dirs
from src.ppe_detection.inference import load_model, predict_image, draw_detections
from src.ppe_detection.ppe_classifier import classify_workers, compliance_color

ensure_dirs()

WEIGHTS    = MODELS_DIR / "yolov8n_ppe_baseline.pt"
# ── UPDATE THIS PATH to your video file ────────────────────────────────────
VIDEO_PATH = Path("/path/to/your/construction_video.mp4")
# ────────────────────────────────────────────────────────────────────────────

model = load_model(WEIGHTS)
print(f"Model loaded: {WEIGHTS}")
print(f"Video path:   {VIDEO_PATH}")
print(f"Video exists: {VIDEO_PATH.exists()}")

## 2. Video Inference with FPS Measurement

We process each frame individually:  
`read frame → predict → classify → draw → write`

FPS is measured as a rolling average over 30 frames.

In [ ]:
def run_full_video(video_path: Path, conf: float = 0.4) -> Path:
    """Run detection + compliance overlay on every frame. Returns output path."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = OUTPUTS_DIR / "videos" / f"ppe_{video_path.stem}.mp4"
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (width, height)
    )

    fps_times: list[float] = []
    frame_idx = 0

    while cap.isOpened():
        t0 = time.perf_counter()
        ret, frame = cap.read()
        if not ret:
            break

        detections = predict_image(model, frame, conf=conf)
        canvas     = draw_detections(frame, detections)

        # Compliance overlay
        workers = classify_workers(detections)
        for w in workers:
            x1, y1, x2, y2 = w.person_bbox
            color = compliance_color(w.status)
            cv2.rectangle(canvas, (x1, y1 - 3), (x2, y1 - 25), color, -1)
            cv2.putText(canvas, w.status.value, (x1 + 4, y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2)

        # FPS overlay
        elapsed = time.perf_counter() - t0
        fps_times.append(elapsed)
        if len(fps_times) > 30:
            fps_times.pop(0)
        live_fps = 1.0 / (sum(fps_times) / len(fps_times))
        cv2.putText(canvas, f"FPS: {live_fps:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        cv2.putText(canvas, f"{frame_idx}/{total}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)

        writer.write(canvas)
        frame_idx += 1
        if frame_idx % 50 == 0:
            print(f"  Frame {frame_idx}/{total} | FPS {live_fps:.1f}")

    cap.release()
    writer.release()
    print(f"\nOutput saved → {out_path}")
    return out_path


if VIDEO_PATH.exists():
    out = run_full_video(VIDEO_PATH, conf=0.4)
else:
    print("Update VIDEO_PATH above to point to a real video file.")

## 3. Preview First Frame

In [ ]:
import matplotlib.pyplot as plt

out_path = OUTPUTS_DIR / "videos" / f"ppe_{VIDEO_PATH.stem}.mp4"

if out_path.exists():
    cap = cv2.VideoCapture(str(out_path))
    ret, frame = cap.read()
    cap.release()
    if ret:
        plt.figure(figsize=(12, 7))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("First Frame — PPE Detection + Compliance Overlay")
        plt.tight_layout()
        preview_path = OUTPUTS_DIR / "images" / "video_preview_frame.png"
        plt.savefig(str(preview_path), dpi=150)
        plt.show()
        print(f"Saved → {preview_path}")
else:
    print("Run the inference cell first.")

## 4. Performance Notes

| Hardware | Expected FPS (YOLOv8n, 640px) |
|----------|-------------------------------|
| RTX 3060 | ~80–120 FPS |
| M1/M2 Mac (MPS) | ~40–60 FPS |
| CPU only | ~5–15 FPS |

For real-time deployment (target: 25+ FPS on camera streams), YOLOv8n on GPU is sufficient.

## 5. Conclusions & Stage 1 Complete

**Stage 1 delivers:**
- A fine-tuned YOLOv8n model detecting 10 PPE classes.
- SAFE/UNSAFE worker classification per frame.
- Inference on images and videos.
- All weights, metrics, and outputs saved and versioned.

**Stage 2** (Vehicle Detection) begins after Stage 1 is validated and committed.